# 数据驻留：锁定智能体的推理地域

涉及受监管数据的智能体在上线之前必须回答一个问题：它的模型请求在何处运行。把答案写在代理层或操作手册里，这项保证就落在执行它的系统之外。

`model.inference_geo` 把答案写进智能体定义。它锁定为智能体模型请求提供服务的地理区域，会对照工作区的驻留策略进行校验，并在每一轮对话中强制执行，因此这一约束随智能体一起迁移，即使策略在智能体上线之后发生变化也依然有效。

本笔记本将依次讲解：

- 把智能体锁定到 `us` 推理地域，并在会话上确认该锁定
- 该锁定如何对照工作区的 `allowed_inference_geos` 和 `default_inference_geo` 进行解析
- 用于清除锁定的更新语义
- 用 `agent_with_overrides` 为单个会话覆盖地域

## 1. 配置客户端

`inference_geo` 依赖标准的 `managed-agents-2026-04-01` 测试版请求头，并且需要 `anthropic>=0.121.0` 才能使用类型化字段。

In [ ]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
client = anthropic.Anthropic()

## 2. 把智能体锁定到美国推理地域

`inference_geo` 位于 `model` 配置内，与 `effort` 和 `speed` 并列。可接受的值是 `"global"` 和 `"us"`。如果不设置，每次模型请求都会按请求被服务那一刻工作区的 `default_inference_geo` 解析，这意味着工作区默认值后来的变更会影响已经在运行的会话。设置之后，锁定就固定在智能体上。

该锁定会被校验三次：保存智能体时、由它创建会话时，以及该会话服务的每一轮对话。它是严格执行的，不做既往豁免。如果工作区管理员事后收窄 `allowed_inference_geos`，来自已不被允许的智能体的新会话会被拒绝，而已经在运行的会话会拒绝它的下一轮对话。这种严格正是要点所在：工作区依赖它实现数据驻留，因此即使会话中途策略变更，锁定依然成立。

In [ ]:
researcher = client.beta.agents.create(
    name="us_records_analyst",
    description="Answers questions about internal records, pinned to US inference.",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-pin-inference-geo"},
    model={"id": MODEL, "inference_geo": "us"},
    system="You answer questions about the records you are given, concisely.",
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)
print(f"{researcher.name}: {researcher.id} v{researcher.version}")
print("inference_geo:", researcher.model.inference_geo)

## 3. 在会话上确认锁定

会话在创建时会快照智能体的配置，因此锁定在 `session.agent.model` 上可见。在那里读回它，就是确认该会话的各轮对话绑定到 `us`，与工作区当前的默认值无关。

In [ ]:
env = client.beta.environments.create(
    name="residency-demo",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent=researcher.id,
    environment_id=env.id,
    title="Records question, US-pinned",
    initial_events=[
        {
            "type": "user.message",
            "content": [
                {"type": "text", "text": "In one sentence: what is a data residency policy?"}
            ],
        }
    ],
    betas=BETAS,
)
print("session pin:", session.agent.model.inference_geo)

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "agent.message":
            print("".join(b.text for b in ev.content if b.type == "text"), end="")
        elif ev.type == "session.status_idle":
            print(f"\n[idle] stop_reason={ev.stop_reason.type}")
            break

## 4. 由工作区决定锁定可以指向什么

工作区的驻留策略位于 Admin API 上，在工作区的 `data_residency` 配置块中：`allowed_inference_geos`（地域列表，或字面量 `"unrestricted"`）和 `default_inference_geo`。除非该字段为 `"unrestricted"`，智能体的 `inference_geo` 必须是 `allowed_inference_geos` 的成员。其他情况在保存时返回 400。在多智能体名册中，锁定必须整体一致：协调者与名册中每个成员要么全部设置为同一地域，要么全部未设置。

有一处值得明确的更新细节。在 `agents.update` 中，`model` 是整体替换而非合并，因此提交不含 `inference_geo` 的 `model` 会清除锁定。若要更换模型 ID 并保留地域，需要重新写明它。

In [ ]:
cleared = client.beta.agents.update(researcher.id, model={"id": MODEL}, betas=BETAS)
print("after model update without geo:", cleared.model.inference_geo)

researcher = client.beta.agents.update(
    researcher.id,
    model={"id": MODEL, "inference_geo": "us"},  # restate the pin
    betas=BETAS,
)
print("restated:", researcher.model.inference_geo)

## 5. 为单个会话覆盖地域

智能体的锁定是由此创建的每个会话的默认值。要在不修改共享智能体的前提下让某个会话运行在不同地域，向 `sessions.create` 传入 `agent_with_overrides` 对象并覆盖 `model`。该覆盖只替换该会话的模型配置，智能体资源不受影响。同样的成员规则仍然适用：覆盖到的地域必须是工作区允许的地域，并且名册中的成员仍须全部一致。

In [ ]:
override = client.beta.sessions.create(
    agent={
        "type": "agent_with_overrides",
        "id": researcher.id,
        "model": {"id": MODEL, "inference_geo": "global"},  # this session only
    },
    environment_id=env.id,
    title="One-off, global geography",
    betas=BETAS,
)
print("override session pin:", override.agent.model.inference_geo)
print(
    "agent still pinned to:",
    client.beta.agents.retrieve(researcher.id, betas=BETAS).model.inference_geo,
)

## 6. 清理

In [ ]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
for s in (session, override):
    client.beta.sessions.archive(s.id, betas=BETAS)
client.beta.agents.archive(researcher.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")

## 锁定的定位

`inference_geo` 是一项数据驻留与合规控制。它不涉及智能体能读取什么或数据存储在何处，只涉及词元在何处被处理。存储地域是一项独立且不可变的工作区设置（`workspace_geo`）；两者结合才能回答合规审查所提出的问题。

因为它在每一轮对话中而非仅在创建时强制执行，所以当保证必须在智能体上线后的变更中依然成立时，锁定就是合适的控制手段。对于大规模部署，应设置工作区的 `default_inference_geo`，让未锁定的智能体继承正确的地域，并把单个智能体的锁定留给那些驻留要求绝不能依赖该默认值的智能体。